<a href="https://colab.research.google.com/github/tanvirRahan/Automated-Certificate-Pipeline/blob/main/certificates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Automated Bulk Certificate Generator Pipeline
**Author:** Tanvir Rahan Rifat  
**Project Type:** ETL Pipeline & Document Automation

This notebook automates the mass generation of professional certificates. It extracts data from structured datasets, dynamically generates role-specific performance summaries, assigns unique tracking IDs, and exports high-quality PDFs using headless LibreOffice conversion.

In [ ]:
# 1. Install Required Dependencies
!pip install -q docxtpl python-docx pandas openpyxl
!apt-get update -qq
!apt-get install -y -qq libreoffice fonts-liberation

# 2. Fetch Templates and Sample Data from GitHub (Using Correct RAW Links)
!wget -q -O sample_data.xlsx https://raw.githubusercontent.com/tanvirRahan/Automated-Certificate-Pipeline/main/sample_data.xlsx
!wget -q -O certificate_template.docx https://raw.githubusercontent.com/tanvirRahan/Automated-Certificate-Pipeline/main/certificate_template.docx

print("✅ Environment Ready: Dependencies installed and GitHub assets downloaded successfully.")

## ⚙️ Core Architecture & Business Logic
Defines dynamic Reference Number generation based on candidate nomenclature and maps industry-standard performance metrics based on specified roles.

In [ ]:
import os
import subprocess
import pandas as pd
from docxtpl import DocxTemplate
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH

TEMPLATE_FILE = 'certificate_template.docx'
DATA_FILE = 'sample_data.xlsx'
OUTPUT_FOLDER = 'Final_Certificates_Batch'

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

def generate_ref_no(full_name):
    words = str(full_name).strip().split()

    if not words:
        return "REF/06/26"

    if len(words) == 1:
        prefix = f"I{words[0][0]}X"
    elif len(words) == 2:
        prefix = f"I{words[0][0]}{words[1][0]}"
    else:
        prefix = "".join(w[0] for w in words)

    return f"{prefix.upper()}/06/26"

def get_certificate_content(interest_raw):
    interest = str(interest_raw).lower().strip()

    if 'frontend' in interest:
        role = "Frontend Developer Intern"
        summary = (
            "Throughout the internship, they demonstrated exceptional proficiency in modern frontend frameworks and responsive architecture. "
            "By translating complex UI/UX wireframes into interactive, high-performance web applications, "
            "they consistently displayed a keen eye for detail, component-driven design, and clean code practices."
        )
    elif 'backend' in interest:
        role = "Backend Developer Intern"
        summary = (
            "Throughout the internship, they showcased a deep understanding of server-side architecture, database management, and robust API development. "
            "They successfully engineered scalable backend solutions, optimized system latency, and demonstrated advanced problem-solving skills, "
            "making them a highly dependable asset to the engineering team."
        )
    elif 'sqa' in interest:
        role = "SQA Engineer Intern"
        summary = (
            "Throughout the internship, they exhibited a meticulous approach to software quality assurance and product lifecycle management. "
            "By designing comprehensive test plans and executing rigorous manual and automated testing protocols, "
            "they significantly reduced defect rates and ensured the delivery of highly stable, production-ready software."
        )
    elif 'ui' in interest or 'ux' in interest or 'design' in interest:
        role = "UI/UX Design Intern"
        summary = (
            "Throughout the internship, they demonstrated profound creativity paired with a highly analytical approach to user-centric design. "
            "By establishing comprehensive wireframes and intuitive prototypes, they successfully translated complex business requirements "
            "into seamless digital experiences, showcasing a strong mastery of modern design principles."
        )
    else:
        role = "Software Engineer Intern"
        summary = (
            "Throughout the internship, they displayed a strong commitment to engineering excellence and the professional execution of all assigned projects. "
            "They consistently demonstrated a proactive technical mindset, highly effective cross-functional communication, "
            "and the ability to seamlessly adapt to emerging technologies and agile methodologies."
        )

    return role, summary

## 🚀 Execution Pipeline
Extracts candidate data, injects parameters into the MS Word template, applies native formatting, and executes headless LibreOffice conversion to finalize PDFs.

In [ ]:
try:
    df = pd.read_excel(DATA_FILE)
    print(f"Dataset loaded successfully. Processing {len(df)} records...\n")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit()

success_count = 0

for index, row in df.iterrows():
    try:
        col_name = 'Full Name' if 'Full Name' in df.columns else df.columns[2]
        col_interest = 'Field of interest' if 'Field of interest' in df.columns else df.columns[7]

        raw_name = str(row[col_name]).strip().title()
        safe_file_name = raw_name.replace(" ", "_")
        interest = str(row[col_interest])

        role, summary = get_certificate_content(interest)
        smart_ref_no = generate_ref_no(raw_name)

        context = {
            'name': raw_name,
            'role': role,
            'performance_summary': summary,
            'ref_no': smart_ref_no
        }

        doc_tpl = DocxTemplate(TEMPLATE_FILE)
        doc_tpl.render(context)

        docx_path = f"{OUTPUT_FOLDER}/{safe_file_name}_Certificate.docx"
        doc_tpl.save(docx_path)

        subprocess.run(
            ['libreoffice', '--headless', '--convert-to', 'pdf', '--outdir', OUTPUT_FOLDER, docx_path],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

        if os.path.exists(docx_path):
            os.remove(docx_path)

        print(f"Generated: {safe_file_name}.pdf | Ref: {smart_ref_no} | Role: {role}")
        success_count += 1

    except Exception as e:
        print(f"Failed processing record {index} ({raw_name}): {e}")

subprocess.run(f"zip -r -q {OUTPUT_FOLDER}.zip {OUTPUT_FOLDER}", shell=True)
print(f"\nPipeline execution completed. {success_count} certificates archived to {OUTPUT_FOLDER}.zip")